In [3]:
!pip install rapidfuzz 

In [7]:
import os
import tkinter as tk
from tkinter import filedialog, messagebox, ttk
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import nltk
from nltk.corpus import stopwords
import string
import fitz  # PyMuPDF
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory
import matplotlib.pyplot as plt
from matplotlib.backends.backend_tkagg import FigureCanvasTkAgg
import re

# Tambahan impor untuk fuzzy matching dan phrase detection
from rapidfuzz import process, fuzz
from nltk.collocations import BigramAssocMeasures, BigramCollocationFinder

# Download resources
nltk.download('stopwords')
nltk.download('punkt')

# Inisialisasi stopwords dan stemmer menggunakan Sastrawi
STOPWORDS = set(stopwords.words('indonesian'))
factory = StemmerFactory()
stemmer = factory.create_stemmer()

# Fungsi untuk ekstraksi teks dari PDF hanya pada judul skripsi
def extract_title_from_pdf(pdf_path):
    doc = fitz.open(pdf_path)
    title = ""
    if len(doc) > 0:
        page = doc[0]
        text_lines = page.get_text("text").split('\n')
        title_candidates = [
            line.strip() for line in text_lines
            if len(line.strip()) > 0 and len(line.split()) >= 3
        ]
        if title_candidates:
            potential_title = ' '.join(title_candidates)
            separators = [" /"]
            for sep in separators:
                if sep in potential_title:
                    potential_title = potential_title.split(sep)[0].strip()
            title = potential_title
    doc.close()
    return title

# Fungsi tambahan untuk perbaikan otomatis
def build_vocabulary(dataset):
    vocab = set()
    for doc in dataset:
        for token in nltk.word_tokenize(doc):
            vocab.add(token)
    return list(vocab)

def normalize_tokens_with_vocab(tokens, vocab, threshold=85):
    normalized = []
    for token in tokens:
        if not vocab:
            normalized.append(token)
            continue
        match, score, _ = process.extractOne(token, vocab, scorer=fuzz.ratio)
        if score >= threshold:
            normalized.append(match)
        else:
            normalized.append(token)
    return normalized

def detect_phrases_in_dataset(tokens, dataset, min_freq=2):
    all_tokens = []
    for doc in dataset:
        all_tokens.extend(nltk.word_tokenize(doc))

    finder = BigramCollocationFinder.from_words(all_tokens)
    finder.apply_freq_filter(min_freq)
    bigram_measures = BigramAssocMeasures()
    top_bigrams = finder.nbest(bigram_measures.pmi, 20)

    phrases = set(["_".join(b) for b in top_bigrams])
    merged_tokens = []
    skip = False
    for i in range(len(tokens)):
        if skip:
            skip = False
            continue
        if i < len(tokens) - 1:
            candidate = tokens[i] + "_" + tokens[i+1]
            if candidate in phrases:
                merged_tokens.append(candidate)
                skip = True
                continue
        merged_tokens.append(tokens[i])
    return merged_tokens

# Fungsi preprocessing teks
def preprocess_text(text, dataset=None, vocab=None):
    text = text.lower()
    tokens = nltk.word_tokenize(text)

    if vocab:
        tokens = normalize_tokens_with_vocab(tokens, vocab)
    if dataset:
        tokens = detect_phrases_in_dataset(tokens, dataset)

    filtered_tokens = [word for word in tokens if word not in STOPWORDS and word not in string.punctuation]
    filtered_tokens = [word for word in filtered_tokens if not word.isdigit()]
    stemmed_tokens = [stemmer.stem(word) for word in filtered_tokens]
    removed_words = [word for word in tokens if word not in filtered_tokens]
    precision = len(stemmed_tokens) / len(tokens) if tokens else 0
    recall = len(stemmed_tokens) / len(filtered_tokens) if filtered_tokens else 0
    return text, ' '.join(stemmed_tokens), removed_words, tokens, stemmed_tokens, precision, recall

# Fungsi untuk memuat dataset PDF
def load_dataset(file_paths, vocab=None, dataset=None):
    data = []
    filenames = []
    for file_path in file_paths:
        raw_text = extract_title_from_pdf(file_path)
        _, preprocessed_text, *_ = preprocess_text(raw_text, dataset, vocab)
        data.append(preprocessed_text)
        filenames.append(os.path.basename(file_path))
    return data, filenames
    
# Fungsi analisis similaritas
def analyze_similarity(new_title, dataset):
    vectorizer = TfidfVectorizer()
    tfidf_matrix = vectorizer.fit_transform(dataset + [new_title])
    cosine_sim = cosine_similarity(tfidf_matrix[-1], tfidf_matrix[:-1])
    return cosine_sim.flatten(), vectorizer.get_feature_names_out(), tfidf_matrix[-1]

# Fungsi untuk menampilkan bobot TF-IDF pada kata dalam judul baru
def display_tfidf_weights(new_title, tfidf_vector, feature_names):
    word_weight_pairs = zip(feature_names, tfidf_vector.toarray()[0])
    word_weight_pairs = sorted(word_weight_pairs, key=lambda x: x[1], reverse=True)
    
    result = []
    new_title_tokens = set(new_title.lower().split())
    
    for word, weight in word_weight_pairs:
        if word in new_title_tokens and weight > 0:
            result.append(f"{word}: {weight:.4f}")
    
    return "\n".join(result)

# GUI
class SimilarityApp:
    def __init__(self, root):
        self.root = root
        self.root.title("Skripsi Similarity Checker")
        self.root.geometry("1200x800")  # Set default window size
        
        self.dataset = []
        self.filenames = []
        self.vocab = []

        self.base_results = []      # list of tuples: [(filename, similarity_float), ...]
        self.last_avg_similarity = 0.0

        # Create main container with paned window for resizable layout
        self.main_paned = ttk.PanedWindow(root, orient=tk.VERTICAL)
        self.main_paned.pack(fill=tk.BOTH, expand=True, padx=10, pady=10)

        # Top frame for controls
        self.top_frame = ttk.Frame(self.main_paned)
        self.main_paned.add(self.top_frame, weight=0)

        # Frame load dataset
        self.load_frame = ttk.LabelFrame(self.top_frame, text="Load Dataset")
        self.load_frame.pack(fill="x", padx=5, pady=5)
        
        self.load_button = ttk.Button(self.load_frame, text="Load PDF Files", command=self.browse_files)
        self.load_button.pack(side="left", padx=5, pady=5)

        # Frame check similarity
        self.check_frame = ttk.LabelFrame(self.top_frame, text="Check Similarity")
        self.check_frame.pack(fill="x", padx=5, pady=5)
        
        self.new_title_label = ttk.Label(self.check_frame, text="Write or Load Your Thesis Title:")
        self.new_title_label.pack(anchor="w", padx=5, pady=2)
        
        self.new_title_entry = ttk.Entry(self.check_frame, width=70)
        self.new_title_entry.pack(fill="x", padx=5, pady=2)
        
        button_frame = ttk.Frame(self.check_frame)
        button_frame.pack(fill="x", padx=5, pady=2)
        
        self.load_new_pdf_button = ttk.Button(button_frame, text="Load Thesis Title", command=self.load_new_pdf)
        self.load_new_pdf_button.pack(side="left", padx=5)
        
        self.check_button = ttk.Button(button_frame, text="Analyze", command=self.check_similarity, style="Analyze.TButton")
        self.check_button.pack(side="right", padx=5)
        
        style = ttk.Style()
        style.configure("Analyze.TButton", foreground="black", background="#4CAF50", font=("Arial", 10, "bold"), padding=4)
        
        # === Frame Sorting Option ===
        self.sort_frame = ttk.LabelFrame(self.top_frame, text="Sorting Options")
        self.sort_frame.pack(fill="x", padx=5, pady=5)

        sort_buttons_frame = ttk.Frame(self.sort_frame)
        sort_buttons_frame.pack(fill="x", padx=5, pady=5)

        self.sort_option = tk.StringVar(value="similarity")
        ttk.Radiobutton(sort_buttons_frame, text="Sort by Similarity (%)",
                        variable=self.sort_option, value="similarity",
                        command=self.on_sort_option_changed).pack(side="left", padx=10)
        ttk.Radiobutton(sort_buttons_frame, text="Sort by Filename (Natural Order)",
                        variable=self.sort_option, value="filename",
                        command=self.on_sort_option_changed).pack(side="left", padx=10)

        # Bottom frame for results with paned window (horizontal split)
        self.bottom_frame = ttk.Frame(self.main_paned)
        self.main_paned.add(self.bottom_frame, weight=3)

        self.result_paned = ttk.PanedWindow(self.bottom_frame, orient=tk.HORIZONTAL)
        self.result_paned.pack(fill=tk.BOTH, expand=True)

        # Left side - Text results
        self.text_frame = ttk.LabelFrame(self.result_paned, text="Analysis Results")
        self.result_paned.add(self.text_frame, weight=1)
        
        # Create scrollable text widget
        text_container = ttk.Frame(self.text_frame)
        text_container.pack(fill=tk.BOTH, expand=True, padx=5, pady=5)
        
        self.result_text = tk.Text(text_container, wrap="word", font=("Arial", 9))
        text_scrollbar_y = ttk.Scrollbar(text_container, orient=tk.VERTICAL, command=self.result_text.yview)
        text_scrollbar_x = ttk.Scrollbar(text_container, orient=tk.HORIZONTAL, command=self.result_text.xview)
        
        self.result_text.configure(yscrollcommand=text_scrollbar_y.set, xscrollcommand=text_scrollbar_x.set)
        
        self.result_text.grid(row=0, column=0, sticky="nsew")
        text_scrollbar_y.grid(row=0, column=1, sticky="ns")
        text_scrollbar_x.grid(row=1, column=0, sticky="ew")
        
        text_container.grid_rowconfigure(0, weight=1)
        text_container.grid_columnconfigure(0, weight=1)
        
        self.result_text.tag_configure("title", font=("Arial", 11, "bold"), foreground="#2A52BE")
        self.result_text.tag_configure("heading", font=("Arial", 9, "bold"), foreground="#FF4500")
        self.result_text.tag_configure("content", font=("Arial", 9), foreground="#333333")
        self.result_text.tag_configure("important", font=("Arial", 9, "italic"), foreground="#228B22")

        # Right side - Chart
        self.chart_frame = ttk.LabelFrame(self.result_paned, text="Similarity Chart")
        self.result_paned.add(self.chart_frame, weight=2)
        
        self.figure = None
        self.canvas = None
        self.chart_container = None

        ### ADD HERE: Save Chart Button (hidden by default)
        self.save_chart_button = ttk.Button(
            self.chart_frame,
            text="Save Chart as Image",
            command=self.save_chart_as_image
        )
        # Jangan dipanggil pack() di sini → nanti setelah chart ada

        # Initialize empty chart
        self.init_empty_chart()

    def init_empty_chart(self):
        """Initialize empty chart placeholder"""
        if self.chart_container:
            self.chart_container.destroy()
            
        self.chart_container = ttk.Frame(self.chart_frame)
        self.chart_container.pack(fill=tk.BOTH, expand=True, padx=5, pady=5)
        
        empty_label = ttk.Label(self.chart_container, text="Chart will appear here after analysis", 
                               font=("Arial", 10), foreground="gray")
        empty_label.place(relx=0.5, rely=0.5, anchor=tk.CENTER)

    def browse_files(self):
        file_selected = filedialog.askopenfilenames(filetypes=[("PDF Files", "*.pdf")])
        if file_selected:
            self.dataset, self.filenames = load_dataset(file_selected, self.vocab, self.dataset)
            self.vocab = build_vocabulary(self.dataset)
            messagebox.showinfo("Success", f"Loaded {len(self.dataset)} PDFs successfully!")
            self.base_results = []
            self.last_avg_similarity = 0.0
            self.init_empty_chart()
        else:
            messagebox.showwarning("No Files Selected", "Please select PDF files to load!")

    def load_new_pdf(self):
        file_selected = filedialog.askopenfilename(filetypes=[("PDF Files", "*.pdf")])
        if file_selected:
            title = extract_title_from_pdf(file_selected)
            self.new_title_entry.delete(0, tk.END)
            self.new_title_entry.insert(0, title)
        else:
            messagebox.showwarning("No File Selected", "Please select a PDF file!")

    def check_similarity(self):
        new_title = self.new_title_entry.get()
        if not new_title.strip():
            messagebox.showerror("Error", "Please enter a new title or load a PDF!")
            return
        if not self.dataset:
            messagebox.showerror("Error", "Dataset is empty. Please load a dataset first!")
            return
        
        _, preprocessed_title, removed_words, tokens, stemmed_tokens, precision, recall = preprocess_text(
            new_title, self.dataset, self.vocab
        )
        
        self.result_text.delete("1.0", tk.END)
        self.result_text.insert(tk.END, "Preprocessing Analysis\n", "title")
        
        self.result_text.insert(tk.END, "\nRemoved Words:\n", "heading")
        self.result_text.insert(tk.END, f"{', '.join(removed_words) if removed_words else 'No words removed.'}\n", "content")
        
        self.result_text.insert(tk.END, "\nStemmed Tokens:\n", "heading")
        self.result_text.insert(tk.END, f"{', '.join(stemmed_tokens)}\n", "important")
        
        self.result_text.insert(tk.END, f"\nPrecision: ", "heading")
        self.result_text.insert(tk.END, f"{precision:.4f}\n", "important")
        self.result_text.insert(tk.END, f"\nRecall: ", "heading")
        self.result_text.insert(tk.END, f"{recall:.4f}\n", "important")
        
        similarities, feature_names, tfidf_vector = analyze_similarity(preprocessed_title, self.dataset)
        avg_similarity = sum(similarities) / len(similarities) * 100
        self.last_avg_similarity = avg_similarity

        tfidf_weights = display_tfidf_weights(preprocessed_title, tfidf_vector, feature_names)
        self.result_text.insert(tk.END, "\nTF-IDF Weights:\n", "heading")
        self.result_text.insert(tk.END, f"{tfidf_weights}\n", "content")
        
        self.base_results = list(zip(self.filenames, similarities))
        self.render_sorted_results()

    def on_sort_option_changed(self):
        if not self.base_results:
            return
        self.render_sorted_results()

    def render_sorted_results(self):
        if self.sort_option.get() == "similarity":
            sorted_results = sorted(self.base_results, key=lambda x: x[1], reverse=True)
        else:
            sorted_results = sorted(self.base_results, key=lambda x: natural_sort_key(x[0]))

        # Update text results
        start_idx = self.result_text.search("Similarity Results:", "1.0", tk.END)
        if start_idx:
            self.result_text.delete(f"{start_idx} linestart", tk.END)

        self.result_text.insert(tk.END, "\nSimilarity Results:\n", "title")
        for filename, similarity in sorted_results:
            self.result_text.insert(tk.END, f"- {filename}: {similarity * 100:.2f}%\n", "content")
        self.result_text.insert(tk.END, f"\nAverage Similarity: {self.last_avg_similarity:.2f}%\n", "important")

        # Update chart
        self.display_bar_chart(sorted_results)
        
        ### ADD HERE: tampilkan tombol save chart setelah chart terbentuk
        if not self.save_chart_button.winfo_ismapped():
            self.save_chart_button.pack(anchor="ne", padx=5, pady=5)

    def display_bar_chart(self, sorted_results):
        """Display responsive bar chart with proper scrolling for large datasets"""
        # Clear previous chart
        if self.chart_container:
            self.chart_container.destroy()
            
        self.chart_container = ttk.Frame(self.chart_frame)
        self.chart_container.pack(fill=tk.BOTH, expand=True, padx=5, pady=5)
        
        if not sorted_results:
            empty_label = ttk.Label(self.chart_container, text="No data to display", 
                                   font=("Arial", 10), foreground="gray")
            empty_label.place(relx=0.5, rely=0.5, anchor=tk.CENTER)
            return

        # Prepare data
        filenames = [f for f, _ in sorted_results]
        scores = [sim * 100 for _, sim in sorted_results]
        
        # Truncate very long filenames for better display
        display_names = []
        for name in filenames:
            if len(name) > 40:  # Limit filename length
                display_names.append(name[:37] + "...")
            else:
                display_names.append(name)

        # Calculate figure dimensions based on data size
        num_items = len(sorted_results)
        bar_height = 0.6  # Height per bar
        min_height = 6
        max_height = 50
        
        # Dynamic height calculation
        calculated_height = max(min_height, min(max_height, num_items * bar_height + 2))
        
        # Create figure with proper DPI for sharp display
        self.figure = plt.Figure(figsize=(10, calculated_height), dpi=100, facecolor='white')
        self.figure.subplots_adjust(left=0.25, right=0.95, top=0.95, bottom=0.08)
        
        ax = self.figure.add_subplot(111)
        
        # Create horizontal bar chart
        y_pos = range(len(display_names))
        bars = ax.barh(y_pos, scores, color='skyblue', alpha=0.8, edgecolor='navy', linewidth=0.5)
        
        # Customize chart
        ax.set_yticks(y_pos)
        ax.set_yticklabels(display_names, fontsize=8)
        ax.set_xlabel('Similarity Percentage (%)', fontsize=10, fontweight='bold')
        ax.set_title(f'Document Similarity Scores (Total: {num_items} documents)', 
                    fontsize=12, fontweight='bold', pad=20)
        ax.invert_yaxis()  # Highest scores at top
        
        # Add value labels on bars
        for i, (bar, score) in enumerate(zip(bars, scores)):
            width = bar.get_width()
            if width > 5:  # Only show label if bar is wide enough
                ax.text(width/2, bar.get_y() + bar.get_height()/2, 
                       f'{score:.1f}%', ha='center', va='center', 
                       fontsize=7, fontweight='bold', color='darkblue')
            else:
                ax.text(width + 0.5, bar.get_y() + bar.get_height()/2, 
                       f'{score:.1f}%', ha='left', va='center', 
                       fontsize=7, fontweight='bold', color='darkblue')
        
        # Set x-axis limits with some padding
        max_score = max(scores) if scores else 100
        ax.set_xlim(0, min(100, max_score * 1.1))
        
        # Add grid for better readability
        ax.grid(axis='x', alpha=0.3, linestyle='--')
        ax.set_axisbelow(True)
        
        # Improve tick formatting
        ax.tick_params(axis='x', labelsize=9)
        ax.tick_params(axis='y', labelsize=8)
        
        # Create scrollable canvas
        canvas_frame = ttk.Frame(self.chart_container)
        canvas_frame.pack(fill=tk.BOTH, expand=True)
        
        # Canvas with scrollbars
        self.canvas_widget = tk.Canvas(canvas_frame, bg='white')
        v_scrollbar = ttk.Scrollbar(canvas_frame, orient=tk.VERTICAL, command=self.canvas_widget.yview)
        h_scrollbar = ttk.Scrollbar(canvas_frame, orient=tk.HORIZONTAL, command=self.canvas_widget.xview)
        
        self.canvas_widget.configure(yscrollcommand=v_scrollbar.set, xscrollcommand=h_scrollbar.set)
        
        # Matplotlib canvas
        self.canvas = FigureCanvasTkAgg(self.figure, master=self.canvas_widget)
        self.canvas.draw()
        
        chart_widget = self.canvas.get_tk_widget()
        
        # Add chart to scrollable canvas
        self.canvas_widget.create_window(0, 0, anchor="nw", window=chart_widget)
        
        # Configure scrolling
        def configure_scroll_region(event=None):
            self.canvas_widget.configure(scrollregion=self.canvas_widget.bbox("all"))
        
        chart_widget.bind('<Configure>', configure_scroll_region)
        
        # Mouse wheel scrolling
        def on_mousewheel(event):
            self.canvas_widget.yview_scroll(int(-1 * (event.delta / 120)), "units")
        
        def on_shift_mousewheel(event):
            self.canvas_widget.xview_scroll(int(-1 * (event.delta / 120)), "units")
        
        # Bind mouse wheel events
        self.canvas_widget.bind("<MouseWheel>", on_mousewheel)
        self.canvas_widget.bind("<Shift-MouseWheel>", on_shift_mousewheel)
        
        # Layout scrollbars and canvas
        self.canvas_widget.grid(row=0, column=0, sticky="nsew")
        v_scrollbar.grid(row=0, column=1, sticky="ns")
        h_scrollbar.grid(row=1, column=0, sticky="ew")
        
        canvas_frame.grid_rowconfigure(0, weight=1)
        canvas_frame.grid_columnconfigure(0, weight=1)
        
        # Update scroll region after everything is packed
        self.root.after(100, configure_scroll_region)

    ### ADD HERE: fungsi simpan chart
    def save_chart_as_image(self):
        """Save current similarity chart as image (PNG/JPG)"""
        if not self.figure:
            messagebox.showerror("Error", "No chart available to save!")
            return

        file_path = filedialog.asksaveasfilename(
            defaultextension=".png",
            filetypes=[("PNG Image", "*.png"), ("JPEG Image", "*.jpg"), ("All Files", "*.*")]
        )
        if file_path:
            try:
                self.figure.savefig(file_path, dpi=300, bbox_inches="tight")
                messagebox.showinfo("Success", f"Chart saved successfully as:\n{file_path}")
            except Exception as e:
                messagebox.showerror("Error", f"Failed to save chart!\n{e}")
    
# Main Program
if __name__ == "__main__":
    root = tk.Tk()
    app = SimilarityApp(root)
    root.mainloop()

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\asus\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\asus\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
